# CrowdSafe AI — MLOps Deployment Report
### Machine Learning 02 - Course Work 2
**HNDDS25.1F**

**Student Index Numbers:** COHNDDS25.1F-024, COHNDDS25.1F-025, COHNDDS25.1F-028


---

This notebook is the continuation of our Deep Learning Course Work (CrowdSafe AI —
a real-time crowd monitoring and risk assessment system using a fine-tuned
YOLOv8 Nano model). This report documents how that model was taken from a
trained `.pt` file to a fully MLOps-managed, deployable, monitored system.

###  All project Documentations, Scripts and needed stuffs are pushed to github . Link is below 

**GitHub Repository:** `https://github.com/DhulakshanKannan/crowdsafe-ai` 


## 1. Problem Definition

### 1.1 Problem Statement
Manual crowd safety monitoring in large public events (stadiums, concerts,
transport hubs) does not scale: human operators suffer fatigue, cannot
watch multiple camera feeds at once, and cannot quantify risk objectively.
There is a need for an automated system that detects people in real time,
quantifies crowd density per zone, and flags dangerous or panic-like
movement before an incident occurs.

CrowdSafe AI addresses this using a fine-tuned YOLOv8 object detection
model combined with a 4×3 grid zone risk engine, a Gaussian density
heatmap, Farneback optical flow for movement direction, and a multi-signal
stampede detector.

### 1.2 Assumptions
- The camera has a reasonably fixed, elevated vantage point over the
  monitored area (typical of CCTV/venue camera placement).
- "Risk" is inferred purely from person count and movement patterns —
  the system does not use facial recognition or any biometric identifiers.
- The deployment environment (this coursework) runs on CPU-only hardware;
  GPU is only used for the one-off training step on Google Colab.

### 1.3 Limitations
- Accuracy degrades in extremely dense crowds (30+ tightly packed people)
  due to bounding-box occlusion — a known limitation of single-stage
  detectors like YOLO in extreme density scenes.
- Optical flow-based movement analysis assumes a mostly static camera;
  a shaking or panning camera would be misread as crowd movement.
- The model was fine-tuned on a Roboflow crowd dataset that, while
  diverse, cannot cover every lighting condition, camera angle, or
  venue type — this is exactly the kind of gap the drift-monitoring
  component (Section 3.4) is designed to catch after deployment.

### 1.4 Dataset Description
Two data sources were used (full detail in the DL coursework report):
1. **COCO pre-training data** — YOLOv8n's original 330k-image, 80-class
   pretrained weights (`person` class reused here).
2. **Fine-tuning dataset** — a Roboflow Universe crowd-detection dataset,
   pre-annotated with person bounding boxes, exported in YOLOv8 format
   (images + label `.txt` files + `data.yaml`), split 70% train / 20%
   validation / 10% test.


## 2. Model Development Pipeline using MLflow 

The training pipeline itself is unchanged from the DL coursework (transfer
learning: YOLOv8n pretrained on COCO → fine-tuned on the Roboflow crowd
dataset). What's new here is wrapping it with **MLflow** so every run's
hyperparameters, metrics, and resulting model file are automatically
tracked and comparable.

### 2.1 Data Preprocessing
| Step | Process | Purpose |
|---|---|---|
| 1 | Frame/Image resize | Roboflow resizes all images to 640×640 to match YOLOv8 input |
| 2 | Auto-orientation correction | Roboflow normalises EXIF rotation |
| 3 | Train/Val/Test split | 70% / 20% / 10% |
| 4 | Class filtering | Only the `person` class retained at inference (class ID 0) |
| 5 | Confidence filtering | Detections below 0.25 confidence discarded at inference time |

### 2.2 Training Pipeline with MLflow
The full script is in `train_mlflow.py` in the project repo. Key excerpt:


In [ ]:
# Excerpt from train_mlflow.py 

import mlflow
from ultralytics import YOLO

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("CrowdSafe-AI-YOLOv8-Finetune")

with mlflow.start_run(run_name="yolov8n_crowd_finetune"):
    mlflow.log_param("epochs", 50)
    mlflow.log_param("imgsz", 640)
    mlflow.log_param("batch", 16)
    mlflow.log_param("patience", 10)

    model = YOLO("yolov8n.pt")
    results = model.train(
        data=r"C:\Users\USER\Desktop\crowdsafe_ai_mlops\crowd-counting-1\data.yaml",
        epochs=50, imgsz=640, batch=16, patience=10, device=0
    )

    metrics = model.val()
    mlflow.log_metric("precision", float(metrics.box.mp))
    mlflow.log_metric("recall", float(metrics.box.mr))
    mlflow.log_metric("mAP50", float(metrics.box.map50))
    mlflow.log_metric("mAP50-95", float(metrics.box.map))

    mlflow.log_artifact("runs/detect/train/weights/best.pt", artifact_path="model")


#### Fig 1: MLflow Dashboard
![mlfd1](ss_images/mlfd1.png)

### 2.3 Why MLflow, and what it gives us
- **Reproducibility:** every run's exact hyperparameters are logged, so any
  training result can be traced back and reproduced.
- **Comparison:** if we fine-tune again later (e.g. with more data), MLflow's
  UI (`mlflow ui`) lets us compare precision/recall/mAP across runs
  side-by-side instead of digging through Colab notebook history.
- **Artifact storage:** the resulting `best.pt` is stored as an MLflow
  artifact linked to the exact run/parameters that produced it — this is
  what "model versioning" means in practice for a DL project.

### 2.4 Training Results (recap from DL coursework)
Training ran for 50 epochs on Google Colab's free T4 GPU, with early
stopping (patience=10) to prevent overfitting. Precision, recall, mAP@0.5
and F1-score were used as evaluation metrics — chosen because false
negatives (missed people) and false positives (phantom detections) both
carry real safety costs in this application.

**Observation:** Logging these metrics via MLflow rather than just printing
them to the Colab console means the result is now a permanent, comparable
artifact rather than something that disappears when the Colab session ends.


#### Fig 2: Overview Details
![mlfd2](ss_images/mlfd2.png)

#### Fig 3: Parameters and Metrics
![mlfd3](ss_images/mlfd3.png)

#### Fig 4 and 5: Model Metrics
![mlfd4](ss_images/mlfd4.png)
![mlfd5](ss_images/mlfd5.png)

## 3. MLOps Implementation

### 3.1 Version Control — Git & GitHub
All code (`app/`, `train_mlflow.py`, `Dockerfile`, workflow files, this
notebook) is tracked in Git and pushed to GitHub:

```bash
git init
git add .
git commit -m "Initial CrowdSafe AI MLOps pipeline"
git remote add origin https://github.com/YOUR_USERNAME/crowdsafe-ai.git
git push -u origin main
```

### 3.2 DVC (Data Version Control) — for large files
Git is not designed for large binary files like datasets and `best.pt`
model weights — committing them bloats the repo and makes diffs useless.
**DVC** solves this by storing a small pointer file in Git while the actual
data/model bytes live in separate remote storage (Google Drive, S3, etc.):

```bash
dvc init
dvc add models/best.pt
git add models/best.pt.dvc .gitignore
git commit -m "Track model weights with DVC"
dvc remote add -d storage gdrive://YOUR_FOLDER_ID  
dvc push
```

This means every new fine-tuned model version gets its own trackable
pointer — we can `dvc checkout` any past model version without bloating Git.
Full explanation in `DVC_AND_VERSIONING_GUIDE.md` in the repo.

### 3.3 CI/CD Pipeline — GitHub Actions + Docker

**GitHub Actions workflow** (`.github/workflows/ci-cd.yml`) runs on every
push: it installs dependencies, runs automated tests (`pytest`), and — only
if tests pass — builds the Docker image. This means a broken model/API
change can never silently reach deployment.


In [ ]:
# Excerpt from .github/workflows/ci-cd.yml

name: CrowdSafe AI CI/CD
on:
  push:
    branches: [ "main" ]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.10" }
      - run: pip install -r requirements.txt
      - run: pytest tests/ -v          # automated testing step

  build-docker:
    needs: test                        
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: docker build -t crowdsafe-ai:${{ '{{ github.sha }}' }} .


**Automated tests** (`tests/test_api.py`) check that:
- the `/health` endpoint responds correctly (used as the deployment
  smoke-test), and
- the `/predict` endpoint fails gracefully (HTTP 400, not a crash) when
  no image is sent.

**Dockerization** — the Flask API is containerized so it runs identically
on any machine (`Dockerfile` in repo root):


#### Fig 6: CrowdSafe AI CI/CD (Main Dashboard)

![cicd1](ss_images/cicd1.png)

#### Fig 7: CrowdSafe AI MLOps Pipeline

![cicd2](ss_images/cicd2.png)

#### Fig 8: Test

![cicd3](ss_images/cicd3.png)

#### Fig 9: Build-Docker

![cicd4](ss_images/cicd4.png)

In [ ]:
# Excerpt from Dockerfile

FROM python:3.10-slim
WORKDIR /app
RUN apt-get update && apt-get install -y libgl1 libglib2.0-0
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ ./app/
COPY models/ ./models/
EXPOSE 5000
HEALTHCHECK --interval=30s CMD python -c "import requests; requests.get('http://localhost:5000/health')"
CMD ["python", "app/api.py"]


Build & run:
```bash
docker build -t crowdsafe-ai:latest .
docker run -p 5000:5000 crowdsafe-ai:latest
curl http://localhost:5000/health
```

### 3.4 Model Deployment — Flask API
The fine-tuned model is deployed as a REST API (`app/api.py`), separate
from the Streamlit dashboard used in the DL coursework demo. This is the
production-facing deployment surface CW2 asks for.

- `GET /health` → `{"status": "ok"}` — used by Docker's HEALTHCHECK and
  the CI pipeline.
- `POST /predict` → accepts an image file, returns JSON with person count,
  bounding boxes, confidence scores, and inference latency.

#### Fig 10: Flask /health check
![fh](ss_images/fh.png)

#### Fig 11: Flask /predict check
![fp](ss_images/fp.png)

### 3.5 Model Monitoring & Logging
Every prediction made through the API is logged to `logs/api_requests.log`
with a timestamp, person count, inference latency, and average detection
confidence (see the `logging` calls inside `app/api.py`). This gives a
persistent, queryable experiment/production log — separate from MLflow,
which tracks *training* runs.

### 3.6 Model Drift Study
`monitor_drift.py` reads those logs and checks for drift using proxy
signals (since there's no ground-truth label available at inference time
in production):

1. **Average detection confidence** — a sustained drop below the
   validation-time baseline suggests the live inputs (new camera angle,
   lighting, unfamiliar crowd density) differ from what the model was
   fine-tuned on.
2. **Inference latency trend** — flags infrastructure issues, not model
   drift per se, but is tracked as part of the same monitoring loop.
3. **Person-count distribution shift** — a sudden change signals the
   deployment context itself changed (e.g. camera moved to a busier area).


In [ ]:
# Excerpt from monitor_drift.py

BASELINE_AVG_CONFIDENCE = 0.75   # from validation metrics logged in MLflow
CONFIDENCE_DROP_ALERT_THRESHOLD = 0.15

recent = df.tail(50)
drift = BASELINE_AVG_CONFIDENCE - recent["avg_confidence"].mean()

if drift > CONFIDENCE_DROP_ALERT_THRESHOLD:
    print("DRIFT ALERT: consider re-collecting data and re-running train_mlflow.py")


**Discussion:** In a classification/regression setting, drift is
usually measured against ground-truth accuracy. A live crowd-monitoring
system has no ground truth available at inference time — nobody is
hand-labelling every frame in production — so this pipeline instead
monitors *proxy signals* (confidence, latency, count distribution) that
correlate with the model seeing out-of-distribution inputs. If drift is
flagged, the recommended response is to collect new labelled frames from
the deployment site and re-run `train_mlflow.py`, producing a new MLflow-
tracked, DVC-versioned model.


## 4. MLOps Workflow — End-to-End Summary

```
 Roboflow Dataset (versioned via DVC)
        │
        ▼
 train_mlflow.py  ── MLflow tracks params + metrics ──▶  mlruns/ (experiment log)
        │
        ▼
 models/best.pt  ── versioned via DVC ──▶  Git repo (.dvc pointer)
        │
        ▼
 git push  ──▶  GitHub Actions CI/CD
        │            │
        │            ├── pytest (automated tests)
        │            └── docker build (only if tests pass)
        ▼
 Docker image  ──▶  docker run  ──▶  Flask API (app/api.py) on :5000
        │
        ▼
 Every /predict request logged  ──▶  logs/api_requests.log
        │
        ▼
 monitor_drift.py  ──▶  drift report + alert if confidence drops
```

**Repository structure:**
```
crowdsafe-ai/
├── app/
│   └── api.py                  # Flask deployment API
├── models/
│   └── best.pt                 # fine-tuned YOLOv8 weights (DVC-tracked)
├── tests/
│   └── test_api.py             # automated tests for CI
├── .github/workflows/
│   └── ci-cd.yml                # GitHub Actions pipeline
├── train_mlflow.py              # MLflow-tracked training pipeline
├── monitor_drift.py             # model monitoring / drift study
├── Dockerfile
├── requirements.txt
├── DVC_AND_VERSIONING_GUIDE.md
└── CW2_ML2_MLOps_Report.ipynb   
```

## Conclusion
This coursework extended the CrowdSafe AI deep learning model (fine-tuned
YOLOv8 for crowd detection) into a fully MLOps-managed system. MLflow
provides reproducible experiment tracking for model training. DVC extends
Git's version control to large model/dataset files. GitHub Actions
automates testing on every push, and Docker guarantees the Flask deployment
runs identically anywhere. Request-level logging combined with the drift
study closes the loop, giving a concrete, actionable signal for when the
model needs to be retrained. Together these practices take the project from
"a trained model that works on my laptop" to a versioned, tested,
containerized, and monitored deployment pipeline.


#### Appendices
![run](ss_images/run.png)
![run2](ss_images/run2.png)

### Thank you 